# 插值与缺失值填补代码示例

插值是根据已知离散点推算中间未知点的值；缺失值填补是在数据预处理阶段用合理方式补上 NaN。

后面的单元格按顺序运行即可。


## 准备工作

先导入会用到的库。后面的单元格按顺序运行即可。


In [1]:
# 导入插值和缺失值填补所需的库。
import numpy as np
import pandas as pd
from scipy import interpolate
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("所有库导入成功")

所有库导入成功


# 一、简单填补缺失值

## 1. 均值、中位数、前向、后向与线性插值填补

| 方法 | 思路 | 适用场景 |
|------|------|---------|
| 均值/中位数 | 用整列统计量填充 | 缺失比例小的快速处理 |
| 前向/后向填充 | 用相邻值替代 | 时间序列，短期缓慢变化 |
| 线性插值填补 | 假设缺失段线性变化 | 中间段缺失，两端有值 |

下面用含 NaN 的序列演示几种常见填补方式。


In [2]:
# 构造含 NaN 的示例数据。
df_example = pd.DataFrame({
    'time': [1, 2, 3, 4, 5, 6, 7, 8],
    'value': [10.0, np.nan, 14.0, np.nan, np.nan, 20.0, 22.0, np.nan]
})
print("原始数据：")
print(df_example)

# 均值填补：用整列平均值填充，简单但降低方差。
df_mean = df_example['value'].fillna(df_example['value'].mean())
print("\n均值填补：")
print(df_mean)

# 中位数填补：不受极端值影响，适合偏态分布。
df_median = df_example['value'].fillna(df_example['value'].median())
print("\n中位数填补：")
print(df_median)

# 前向填充：用上一个非 NaN 值填充，适合缓慢变化的时间序列。
df_ffill = df_example['value'].ffill()
print("\n前向填充：")
print(df_ffill)

# 后向填充：从未来值向前推，末尾 NaN 无法填充。
df_bfill = df_example['value'].bfill()
print("\n后向填充：")
print(df_bfill)

# 线性插值填补：假设缺失段前后线性变化。
df_lin = df_example['value'].interpolate(method='linear')
print("\n线性插值填补：")
print(df_lin)

原始数据：
   time  value
0     1   10.0
1     2    NaN
2     3   14.0
3     4    NaN
4     5    NaN
5     6   20.0
6     7   22.0
7     8    NaN

均值填补：
0    10.0
1    16.5
2    14.0
3    16.5
4    16.5
5    20.0
6    22.0
7    16.5
Name: value, dtype: float64

中位数填补：
0    10.0
1    17.0
2    14.0
3    17.0
4    17.0
5    20.0
6    22.0
7    17.0
Name: value, dtype: float64

前向填充：
0    10.0
1    10.0
2    14.0
3    14.0
4    14.0
5    20.0
6    22.0
7    22.0
Name: value, dtype: float64

后向填充：
0    10.0
1    14.0
2    14.0
3    20.0
4    20.0
5    20.0
6    22.0
7     NaN
Name: value, dtype: float64

线性插值填补：
0    10.0
1    12.0
2    14.0
3    16.0
4    18.0
5    20.0
6    22.0
7    22.0
Name: value, dtype: float64


# 二、一维插值

## 1. 线性、三次样条和最近邻插值

`scipy.interpolate.interp1d` 根据已知散点 $(x_i,y_i)$ 构造插值函数 $f$，用 $f$ 求任意 $x$ 处的 $y$ 值。

线性插值假设相邻点之间是直线；三次样条用分段三次多项式使整条曲线光滑（一阶和二阶导数均连续）；最近邻取最近的已知点值。

- **输入**：x 和 y 的一维数组，以及待求的 x_query
- **输出**：插值函数对象，可对单值或数组计算
- **优点**：接口统一，支持多种方法
- **缺点**：不能外推（除非显式指定 fill_value='extrapolate'）


In [3]:
# 构造示例数据，模拟某检测值随时间的变化。
x_known = np.array([1.0, 3.0, 5.0, 7.0, 9.0])
y_known = np.array([2.5, 3.1, 4.8, 6.2, 6.0])
x_query = np.array([4.0, 6.0])

# 线性插值：计算简单，不产生过冲，但在节点处有尖角。
f_linear = interpolate.interp1d(x_known, y_known, kind='linear', bounds_error=False, fill_value=np.nan)
print("线性插值：", np.round(f_linear(x_query), 4))

# 三次样条插值：曲线光滑，适合论文配图，稀疏数据处可能振荡。
f_cubic = interpolate.interp1d(x_known, y_known, kind='cubic', bounds_error=False, fill_value=np.nan)
print("三次样条：", np.round(f_cubic(x_query), 4))

# 最近邻插值：值一定是原始数据中出现过的，适合分类变量。
f_nearest = interpolate.interp1d(x_known, y_known, kind='nearest', bounds_error=False, fill_value=np.nan)
print("最近邻插值：", np.round(f_nearest(x_query), 4))

# 生成密集时间点用于光滑绘图。
x_dense = np.linspace(1, 9, 50)
y_linear_dense = f_linear(x_dense)
y_cubic_dense = f_cubic(x_dense)

线性插值： [3.95 5.5 ]
三次样条： [3.9016 5.6203]
最近邻插值： [3.1 4.8]


# 三、二维散点插值

## 1. griddata 与 RBFInterpolator

对不规则分布的二维散点插值到规则网格。常用于地理等值线和实验响应曲面。

- **输入**：points (N,2) 坐标，values (N,) 值，目标网格坐标
- **输出**：目标网格上的插值结果
- **griddata**：支持 linear / cubic / nearest，不能外推
- **RBFInterpolator**：可外推，精度高，但计算量为 $O(N^3)$


In [4]:
# 构造 20 个随机点，z = x*exp(-x^2 - y^2)。
np.random.seed(1)
n_points = 20
points = np.random.rand(n_points, 2) * 4 - 2
values = points[:, 0] * np.exp(-points[:, 0]**2 - points[:, 1]**2)

# 定义目标网格。
grid_x, grid_y = np.meshgrid(np.linspace(-2, 2, 10), np.linspace(-2, 2, 10))
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

print(f"已知散点数: {n_points}，目标网格: {grid_x.size} 个点")

# griddata: 接口统一，不能外推，立方方法在边界不稳定。
for method in ['linear', 'cubic', 'nearest']:
    z_grid = interpolate.griddata(points, values, (grid_x, grid_y), method=method)
    n_valid = np.sum(~np.isnan(z_grid))
    print(f"  griddata({method}): 有效网格点数 = {n_valid}/{grid_x.size}")

# RBFInterpolator: 可以外推，精度高，但计算量大。
rbf = interpolate.RBFInterpolator(points, values, kernel='thin_plate_spline', smoothing=0)
z_rbf = rbf(grid_points).reshape(grid_x.shape)
print(f"  RBFInterpolator: 网格计算完成，形状 = {z_rbf.shape}")

已知散点数: 20，目标网格: 100 个点
  griddata(linear): 有效网格点数 = 48/100
  griddata(cubic): 有效网格点数 = 48/100
  griddata(nearest): 有效网格点数 = 100/100
  RBFInterpolator: 网格计算完成，形状 = (10, 10)


# 四、高级缺失值填补

## 1. SimpleImputer、KNNImputer 与 IterativeImputer

| 方法 | 原理 | 适用场景 |
|------|------|---------|
| SimpleImputer | 每列独立用均值/中位数填充 | 快速基线处理 |
| KNNImputer | 用 K 个近邻的加权均值填补 | 样本充足，特征维度 < 100 |
| IterativeImputer | 轮流将各列作为因变量回归预测 | 精度优先，特征关系复杂 |


In [5]:
# 构造含 NaN 的 3 维特征矩阵。
X_missing = np.array([
    [1.0, 2.0, 3.0],
    [np.nan, 5.0, 6.0],
    [7.0, np.nan, 9.0],
    [10.0, 11.0, np.nan],
    [4.0, np.nan, np.nan],
    [13.0, 14.0, 15.0]
])
print("原始数据（含 NaN）：\n", X_missing)

# SimpleImputer: 每列独立处理，忽略变量间关系。
si_mean = SimpleImputer(strategy='mean')
print("\nSimpleImputer(mean):\n", si_mean.fit_transform(X_missing))

si_median = SimpleImputer(strategy='median')
print("\nSimpleImputer(median):\n", si_median.fit_transform(X_missing))

# KNNImputer: 利用样本间相似性填补，需要调 K 值。
knn = KNNImputer(n_neighbors=2, weights='uniform')
print("\nKNNImputer(k=2):\n", knn.fit_transform(X_missing))

# IterativeImputer: 用回归模型迭代预测缺失值，精度高但计算量大。
iter_imp = IterativeImputer(max_iter=10, random_state=42)
print("\nIterativeImputer:\n", iter_imp.fit_transform(X_missing))

原始数据（含 NaN）：
 [[ 1.  2.  3.]
 [nan  5.  6.]
 [ 7. nan  9.]
 [10. 11. nan]
 [ 4. nan nan]
 [13. 14. 15.]]

SimpleImputer(mean):
 [[ 1.    2.    3.  ]
 [ 7.    5.    6.  ]
 [ 7.    8.    9.  ]
 [10.   11.    8.25]
 [ 4.    8.    8.25]
 [13.   14.   15.  ]]

SimpleImputer(median):
 [[ 1.   2.   3. ]
 [ 7.   5.   6. ]
 [ 7.   8.   9. ]
 [10.  11.   7.5]
 [ 4.   8.   7.5]
 [13.  14.  15. ]]

KNNImputer(k=2):
 [[ 1.   2.   3. ]
 [ 4.   5.   6. ]
 [ 7.   8.   9. ]
 [10.  11.  12. ]
 [ 4.   6.5  6. ]
 [13.  14.  15. ]]

IterativeImputer:
 [[ 1.          2.          3.        ]
 [ 3.95116925  5.          6.        ]
 [ 7.          8.00010652  9.        ]
 [10.         11.         11.99997302]
 [ 4.          5.16415072  6.16405073]
 [13.         14.         15.        ]]


# 五、批量插值（纵向数据对齐）

对多名患者的不规则观测，先定义统一时间网格，再用 `groupby` 对每人分别做线性插值，最后对齐到宽表。

- **输入**：DataFrame 含组标识列、时间列、值列
- **输出**：宽表，行 = 各对象，列 = 统一时点
- **优点**：保留个体趋势，统一网格后便于建模
- **缺点**：每组至少需要 2 个非缺失点；超出范围返回 NaN


In [6]:
# 构造 5 名患者的不规则随访数据。
np.random.seed(42)
patient_ids, times, values = [], [], []
true_trend = {'A': 0.5, 'B': 0.3, 'C': 0.7, 'D': 0.2, 'E': 0.9}
for pid, base in true_trend.items():
    n_obs = np.random.randint(3, 6)
    t = np.sort(np.random.uniform(0, 10, n_obs))
    v = base * t + np.random.normal(0, 0.2, n_obs)
    patient_ids.extend([pid] * n_obs)
    times.extend(t.tolist())
    values.extend(v.tolist())

df_long = pd.DataFrame({'patient': patient_ids, 'time': times, 'value': values})
print("纵向数据（部分）：")
print(df_long.head(12).to_string(index=False))

# 定义统一时间网格。
uniform_grid = np.arange(0, 11, 1)
print("\n统一时间网格:", uniform_grid)

# 对单个对象插值的函数。
def interpolate_group(g):
    x, y = g['time'].values, g['value'].values
    f = interpolate.interp1d(x, y, kind='linear', bounds_error=False, fill_value=np.nan)
    return pd.Series(f(uniform_grid), index=uniform_grid)

# 按患者分组插值，得到宽表。
df_wide = df_long.groupby('patient').apply(interpolate_group)
print("\n批量插值结果（行 = 患者，列 = 统一时点）：")
print(df_wide.round(4).to_string())
print(f"\n结果形状: {df_wide.shape[0]} 人 × {df_wide.shape[1]} 个时点")

纵向数据（部分）：
patient     time    value
      A 1.834348 0.898250
      A 4.458328 2.043398
      A 5.968502 2.807205
      A 7.796910 3.816017
      A 7.965430 3.886191
      B 1.818250 0.578308
      B 1.834045 0.027704
      B 2.123391 0.827091
      B 3.042422 1.076016
      C 0.466657 0.021884
      C 0.906064 0.338979
      C 2.327713 1.926796

统一时间网格: [ 0  1  2  3  4  5  6  7  8  9 10]

批量插值结果（行 = 患者，列 = 统一时点）：
         0       1       2       3       4       5       6       7      8       9   10
patient                                                                               
A       NaN     NaN  0.9705  1.4070  1.8434  2.3174  2.8246  3.3763    NaN     NaN NaN
B       NaN     NaN  0.4862  1.0645     NaN     NaN     NaN     NaN    NaN     NaN NaN
C       NaN  0.4439  1.5608  2.3700  3.0292  3.6884  4.3476  5.0068  5.666  6.3252 NaN
D       NaN  0.2078  0.3008  0.5717  0.8425  1.1133  1.3841     NaN    NaN     NaN NaN
E       NaN     NaN     NaN  2.7544  3.6200  4.4857  5.3513 